In [2]:
from IPython.display import HTML
HTML('''
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>Real-Time 3D Wealth Waveform Motion Model</title>
<style>
  :root {
    --bg: #06101f;
    --panel: rgba(8, 18, 34, 0.78);
    --grid: rgba(120, 180, 255, 0.22);
    --grid-strong: rgba(170, 220, 255, 0.56);
    --text: rgba(245, 250, 255, 0.95);
    --muted: rgba(210, 230, 255, 0.72);
    --blue: #35a6ff;
    --orange: #ff9c27;
    --green: #59ff67;
  }
  * { box-sizing: border-box; }
  body {
    margin: 0;
    min-height: 100vh;
    background:
      radial-gradient(circle at 50% 20%, rgba(38, 99, 174, .28), transparent 42%),
      radial-gradient(circle at 80% 70%, rgba(0, 255, 130, .10), transparent 30%),
      linear-gradient(145deg, #020813 0%, #06101f 60%, #02050a 100%);
    font-family: Inter, Segoe UI, Roboto, Arial, sans-serif;
    color: var(--text);
    overflow: hidden;
  }
  .wrap {
    display: grid;
    grid-template-rows: auto 1fr auto;
    height: 100vh;
    padding: 22px;
    gap: 14px;
  }
  header {
    text-align: center;
    letter-spacing: .04em;
  }
  h1 {
    margin: 0;
    font-size: clamp(28px, 5vw, 54px);
    font-weight: 800;
    text-shadow: 0 0 18px rgba(255,255,255,.45), 0 0 40px rgba(73,165,255,.22);
  }
  .subtitle {
    margin-top: 6px;
    color: var(--muted);
    font-size: 14px;
  }
  .stage {
    position: relative;
    border: 1px solid rgba(145, 205, 255, .25);
    border-radius: 24px;
    overflow: hidden;
    background:
      linear-gradient(180deg, rgba(255,255,255,.05), rgba(255,255,255,.01)),
      rgba(3, 11, 24, .38);
    box-shadow:
      0 25px 80px rgba(0,0,0,.45),
      inset 0 0 38px rgba(65, 151, 255, .12);
  }
  canvas {
    display: block;
    width: 100%;
    height: 100%;
  }
  .legend {
    position: absolute;
    left: 22px;
    bottom: 22px;
    padding: 14px 16px;
    border-radius: 14px;
    background: var(--panel);
    border: 1px solid rgba(180,220,255,.32);
    box-shadow: 0 0 28px rgba(40,140,255,.15);
    backdrop-filter: blur(8px);
    min-width: 300px;
  }
  .legend-row {
    display: flex;
    align-items: center;
    gap: 12px;
    margin: 9px 0;
    color: var(--text);
    font-weight: 600;
  }
  .sample {
    width: 58px;
    height: 0;
    border-top-width: 4px;
    border-top-style: solid;
    filter: drop-shadow(0 0 6px currentColor);
  }
  .blue { color: var(--blue); border-top-style: dashed; }
  .orange { color: var(--orange); border-top-style: dotted; border-top-width: 5px; }
  .green { color: var(--green); }
  .hud {
    position: absolute;
    top: 22px;
    right: 22px;
    padding: 12px 14px;
    border-radius: 14px;
    background: var(--panel);
    border: 1px solid rgba(180,220,255,.25);
    color: var(--muted);
    font-size: 13px;
    line-height: 1.45;
    backdrop-filter: blur(8px);
  }
  .controls {
    display: flex;
    gap: 12px;
    align-items: center;
    justify-content: center;
    flex-wrap: wrap;
  }
  button, input[type="range"] {
    accent-color: #56a9ff;
  }
  button {
    border: 1px solid rgba(180,220,255,.28);
    color: var(--text);
    background: rgba(12, 30, 55, .84);
    border-radius: 999px;
    padding: 10px 16px;
    font-weight: 700;
    cursor: pointer;
    box-shadow: 0 0 20px rgba(46, 135, 255, .12);
  }
  button:hover { background: rgba(18, 45, 80, .95); }
  label {
    color: var(--muted);
    font-weight: 600;
    display: flex;
    gap: 8px;
    align-items: center;
  }
</style>
</head>
<body>
  <div class="wrap">
    <header>
      <h1>demo</h1>
      <div class="subtitle">Real-time 3D motion simulation of original, target, and transferred wealth waveforms</div>
    </header>

    <main class="stage">
      <canvas id="sim"></canvas>

      <div class="hud">
        <div><strong>Live model state</strong></div>
        <div id="timeReadout">Iteration: 0.0</div>
        <div id="transferReadout">Transfer phase: baseline</div>
      </div>

      <div class="legend">
        <div class="legend-row"><span class="sample blue"></span>Original Wealth Waveform</div>
        <div class="legend-row"><span class="sample orange"></span>Target Account</div>
        <div class="legend-row"><span class="sample green"></span>Transferred Wealth Waveform</div>
      </div>
    </main>

    <section class="controls">
      <button id="pause">Pause</button>
      <button id="reset">Reset</button>
      <label>Speed <input id="speed" type="range" min="0.2" max="3" step="0.1" value="1"></label>
      <label>Glow <input id="glow" type="range" min="2" max="22" step="1" value="12"></label>
    </section>
  </div>

<script>
const canvas = document.getElementById("sim");
const ctx = canvas.getContext("2d");
const pauseBtn = document.getElementById("pause");
const resetBtn = document.getElementById("reset");
const speedSlider = document.getElementById("speed");
const glowSlider = document.getElementById("glow");
const timeReadout = document.getElementById("timeReadout");
const transferReadout = document.getElementById("transferReadout");

let W, H, dpr;
let running = true;
let t0 = performance.now();
let phaseOffset = 0;

function resize() {
  dpr = Math.max(1, Math.min(2, window.devicePixelRatio || 1));
  W = canvas.clientWidth;
  H = canvas.clientHeight;
  canvas.width = Math.floor(W * dpr);
  canvas.height = Math.floor(H * dpr);
  ctx.setTransform(dpr, 0, 0, dpr, 0, 0);
}
window.addEventListener("resize", resize);
resize();

const COLORS = {
  blue: "#35a6ff",
  orange: "#ff9c27",
  green: "#59ff67",
  grid: "rgba(140, 195, 255, .22)",
  gridStrong: "rgba(185, 225, 255, .52)",
  text: "rgba(245,250,255,.95)",
  muted: "rgba(214,232,255,.75)"
};

function project(p) {
  // Isometric-like 3D projection. p = {x:0..100, y:0..2, z:-1.1..1.1}
  const sx = W * 0.15;
  const sy = H * 0.80;
  const scaleX = W * 0.0069;
  const scaleY = W * 0.14;
  const scaleZ = H * 0.26;

  const px = sx + p.x * scaleX + p.y * scaleY;
  const py = sy - p.x * scaleX * 0.34 - p.y * scaleY * 0.24 - p.z * scaleZ;
  return {x: px, y: py};
}

function original(x, elapsed) {
  return Math.sin((x / 100) * Math.PI * 2 + elapsed * 0.55) * 0.98;
}

function target(x) {
  return 0.06 - x * 0.0026;
}

function transferred(x, elapsed) {
  // Baseline with pulsing transition; step arrives near x=82 and travels slightly over time.
  const travel = 78 + 8 * Math.sin(elapsed * 0.55);
  const step = 1 / (1 + Math.exp(-(x - travel) * 1.8));
  const baseline = 0.34 - x * 0.0022;
  return baseline * (1 - step) + 0.98 * step;
}

function drawText(text, x, y, size = 14, align = "center", rotation = 0, weight = 700) {
  ctx.save();
  ctx.translate(x, y);
  ctx.rotate(rotation);
  ctx.font = `${weight} ${size}px Inter, Segoe UI, Arial, sans-serif`;
  ctx.textAlign = align;
  ctx.textBaseline = "middle";
  ctx.fillStyle = COLORS.text;
  ctx.shadowColor = "rgba(150,210,255,.45)";
  ctx.shadowBlur = 8;
  ctx.fillText(text, 0, 0);
  ctx.restore();
}

function line3D(points, color, width = 3, dash = [], glow = 12, progress = 1) {
  const n = Math.max(2, Math.floor(points.length * progress));
  ctx.save();
  ctx.strokeStyle = color;
  ctx.lineWidth = width;
  ctx.setLineDash(dash);
  ctx.lineJoin = "round";
  ctx.lineCap = "round";
  ctx.shadowColor = color;
  ctx.shadowBlur = glow;
  ctx.beginPath();
  for (let i = 0; i < n; i++) {
    const q = project(points[i]);
    if (i === 0) ctx.moveTo(q.x, q.y);
    else ctx.lineTo(q.x, q.y);
  }
  ctx.stroke();

  // Secondary core line for crispness
  ctx.shadowBlur = 0;
  ctx.globalAlpha = 0.82;
  ctx.lineWidth = Math.max(1.2, width * 0.42);
  ctx.stroke();
  ctx.restore();
}

function dotPulse(p, color, radius = 6, glow = 18) {
  const q = project(p);
  ctx.save();
  ctx.fillStyle = color;
  ctx.shadowColor = color;
  ctx.shadowBlur = glow;
  ctx.beginPath();
  ctx.arc(q.x, q.y, radius, 0, Math.PI * 2);
  ctx.fill();
  ctx.restore();
}

function drawGrid() {
  ctx.save();

  // Floor and back grid
  for (let x = 0; x <= 100; x += 10) {
    const a = project({x, y: 0, z: -1.1});
    const b = project({x, y: 2, z: -1.1});
    const c = project({x, y: 2, z: 1.1});
    ctx.strokeStyle = x % 20 === 0 ? COLORS.gridStrong : COLORS.grid;
    ctx.lineWidth = x % 20 === 0 ? 1.2 : 0.7;

    ctx.beginPath(); ctx.moveTo(a.x, a.y); ctx.lineTo(b.x, b.y); ctx.stroke();
    ctx.beginPath(); ctx.moveTo(b.x, b.y); ctx.lineTo(c.x, c.y); ctx.stroke();
  }
  for (let y = 0; y <= 2.0001; y += .5) {
    const a = project({x: 0, y, z: -1.1});
    const b = project({x: 100, y, z: -1.1});
    ctx.strokeStyle = COLORS.grid;
    ctx.beginPath(); ctx.moveTo(a.x, a.y); ctx.lineTo(b.x, b.y); ctx.stroke();
  }
  for (let z = -1; z <= 1.0001; z += .25) {
    const a = project({x: 0, y: 2, z});
    const b = project({x: 100, y: 2, z});
    ctx.strokeStyle = Math.abs(z) < .01 ? COLORS.gridStrong : COLORS.grid;
    ctx.beginPath(); ctx.moveTo(a.x, a.y); ctx.lineTo(b.x, b.y); ctx.stroke();

    const c = project({x: 0, y: 0, z});
    const d = project({x: 0, y: 2, z});
    ctx.beginPath(); ctx.moveTo(c.x, c.y); ctx.lineTo(d.x, d.y); ctx.stroke();
  }

  // Axis edges
  const edges = [
    [{x:0,y:0,z:-1.1},{x:100,y:0,z:-1.1}],
    [{x:0,y:0,z:-1.1},{x:0,y:2,z:-1.1}],
    [{x:0,y:0,z:-1.1},{x:0,y:0,z:1.1}],
    [{x:100,y:0,z:-1.1},{x:100,y:2,z:-1.1}],
    [{x:100,y:2,z:-1.1},{x:100,y:2,z:1.1}],
    [{x:0,y:2,z:1.1},{x:100,y:2,z:1.1}],
  ];
  ctx.strokeStyle = "rgba(215,240,255,.72)";
  ctx.shadowColor = "rgba(100,180,255,.7)";
  ctx.shadowBlur = 8;
  ctx.lineWidth = 1.6;
  for (const [a,b] of edges) {
    const pa = project(a), pb = project(b);
    ctx.beginPath(); ctx.moveTo(pa.x, pa.y); ctx.lineTo(pb.x, pb.y); ctx.stroke();
  }

  // Axis labels and ticks
  [0,20,40,60,80,100].forEach(x => {
    const p = project({x, y:0, z:-1.1});
    drawText(String(x), p.x, p.y + 22, 15);
  });
  [-1, -.75, -.5, -.25, 0, .25, .5, .75, 1].forEach(z => {
    const p = project({x:0, y:0, z});
    drawText(z.toFixed(2), p.x - 34, p.y, 14, "right");
  });

  const xLabel = project({x:55,y:0,z:-1.1});
  drawText("Iterations (Time)", xLabel.x + 10, xLabel.y + 58, 22, "center", 0.02, 800);

  const zLabel = project({x:0,y:0,z:0});
  drawText("Amplitude", zLabel.x - 92, zLabel.y, 22, "center", -Math.PI/2, 800);

  const cat = [
    ["Original Waveform", 0],
    ["Target Account", 1],
    ["Transferred Waveform", 2],
  ];
  cat.forEach(([label, y]) => {
    const p = project({x:104, y, z:-1.1});
    drawText(label, p.x + 8, p.y, 14, "left", 0, 750);
  });
  const yLabel = project({x:108,y:1,z:-1.1});
  drawText("Waveform Type", yLabel.x + 76, yLabel.y + 36, 22, "center", -0.95, 800);

  ctx.restore();
}

function drawScene(now) {
  const speed = Number(speedSlider.value);
  const elapsed = ((now - t0) / 1000) * speed + phaseOffset;
  const glow = Number(glowSlider.value);

  ctx.clearRect(0, 0, W, H);

  // subtle stars / scanning lines
  ctx.save();
  ctx.globalAlpha = .16;
  for (let i = 0; i < 45; i++) {
    const x = (i * 211 + elapsed * 18) % W;
    const y = (i * 97) % H;
    ctx.fillStyle = "white";
    ctx.fillRect(x, y, 1.2, 1.2);
  }
  ctx.restore();

  drawGrid();

  const blue = [], orange = [], green = [];
  for (let x = 0; x <= 100; x += 1) {
    blue.push({x, y:0, z:original(x, elapsed)});
    orange.push({x, y:1, z:target(x)});
    green.push({x, y:2, z:transferred(x, elapsed)});
  }

  const drawProgress = (elapsed * 0.18) % 1;
  const progress = 0.25 + 0.75 * drawProgress;

  // ghost trails
  ctx.globalAlpha = .20;
  for (let k = 1; k <= 3; k++) {
    const old = elapsed - k * .22;
    const trailBlue = [], trailGreen = [];
    for (let x = 0; x <= 100; x += 2) {
      trailBlue.push({x, y:0, z:original(x, old)});
      trailGreen.push({x, y:2, z:transferred(x, old)});
    }
    line3D(trailBlue, COLORS.blue, 2, [8,8], glow * .7, 1);
    line3D(trailGreen, COLORS.green, 2, [], glow * .7, 1);
  }
  ctx.globalAlpha = 1;

  line3D(blue, COLORS.blue, 4, [9, 8], glow, progress);
  line3D(orange, COLORS.orange, 4, [1, 9], glow * .85, 1);
  line3D(green, COLORS.green, 4.5, [], glow + 4, progress);

  // moving energy particles
  const ix = (elapsed * 24) % 100;
  dotPulse({x: ix, y:0, z:original(ix, elapsed)}, COLORS.blue, 5.5, glow + 8);
  dotPulse({x: ix, y:1, z:target(ix)}, COLORS.orange, 4.5, glow + 4);
  dotPulse({x: ix, y:2, z:transferred(ix, elapsed)}, COLORS.green, 6.2, glow + 10);

  // pulse at transfer step
  const stepX = 78 + 8 * Math.sin(elapsed * 0.55);
  dotPulse({x: stepX, y:2, z:transferred(stepX, elapsed)}, COLORS.green, 9 + 2*Math.sin(elapsed*8), glow + 18);

  timeReadout.textContent = `Iteration: ${ix.toFixed(1)} / 100`;
  transferReadout.textContent = ix > stepX ? "Transfer phase: amplified plateau" : "Transfer phase: baseline accumulation";

  if (running) requestAnimationFrame(drawScene);
}
requestAnimationFrame(drawScene);

pauseBtn.addEventListener("click", () => {
  running = !running;
  pauseBtn.textContent = running ? "Pause" : "Resume";
  if (running) {
    t0 = performance.now();
    requestAnimationFrame(drawScene);
  } else {
    phaseOffset += ((performance.now() - t0) / 1000) * Number(speedSlider.value);
  }
});

resetBtn.addEventListener("click", () => {
  phaseOffset = 0;
  t0 = performance.now();
  if (!running) {
    running = true;
    pauseBtn.textContent = "Pause";
    requestAnimationFrame(drawScene);
  }
});
</script>
</body>
</html>
''')
